In [ ]:
%pip install langchain langchain-google-genai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
load_dotenv()
print("Clave encontrada:", bool(os.getenv("GOOGLE_API_KEY")))

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=1)
print(llm.invoke("Responde solo: hola").content)

In [ ]:
def texto(msg):
    c = msg.content
    if isinstance(c, str):
        return c
    return "".join(b.get("text", "") for b in c if isinstance(b, dict) and b.get("type") == "text")

print(texto(llm.invoke("Responde solo: hola")))   # debe imprimir: hola

In [ ]:
from pathlib import Path
RAIZ = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
print(list(RAIZ.rglob("*.db")) + list(RAIZ.rglob("*.sqlite")))

In [ ]:
import sqlite3
from langchain.tools import tool

RUTA_DB = RAIZ / "data" / "ruta"   # <- ajusta con la ruta encontrada

@tool
def consultar_fallas(tipo: str) -> str:
    """Devuelve la tasa de falla de las máquinas según su tipo de calidad (L, M o H).
    Úsala para preguntas numéricas sobre la base de datos de lecturas de sensores:
    cuántas lecturas hay, cuántas fallas y qué porcentaje de falla tiene un tipo."""
    tipo = tipo.strip().upper()
    if tipo not in ("L", "M", "H"):
        return "Tipo no válido. Los tipos disponibles son L, M y H."
    try:
        with sqlite3.connect(RUTA_DB) as con:
            filas, fallas = con.execute(
                "SELECT COUNT(*), SUM(falla) FROM lecturas WHERE tipo = ?", (tipo,)
            ).fetchone()
        return f"Tipo {tipo}: {filas} lecturas, {fallas} fallas, tasa de falla {100 * fallas / filas:.2f} %."
    except Exception as e:
        return f"Error al consultar la base de datos: {e}"

print(consultar_fallas.invoke({"tipo": "L"}))


In [ ]:
import sys
from pathlib import Path

RAG_SRC = Path(r"ruta")
sys.path.insert(0, str(RAG_SRC))

print("Carpeta src existe:", RAG_SRC.exists())
print("chroma_db existe:  ", (RAG_SRC.parent / "chroma_db").exists())

In [ ]:
from langchain.tools import tool
from recuperar import recuperar, armar_contexto

UMBRAL = 0.65   # igual que en asistente.py
K = 8

@tool
def buscar_documentos(pregunta: str) -> str:
    """Busca información en los documentos cargados (CANSAT AIR-SAT ENMICE-2025 ENTREGABLE, DOCUMENTO_SEGUIDOR_DE_LINEA, PROYECTO ECUACIONES DIFERENCIALES MODELACION DE TERREMOTO, PROYECTO TRANSFORMADA DE LAPLACE). Úsala para preguntas cuyo contenido
    esté en esos documentos: descripciones, componentes, conceptos, explicaciones.
    Devuelve fragmentos con su archivo y página. No sirve para datos numéricos
    de fallas de máquinas."""
    try:
        fragmentos = recuperar(pregunta, k=K)
        if not fragmentos or fragmentos[0]["distancia"] > UMBRAL:
            return "No se encontró información relevante en los documentos."
        return armar_contexto(fragmentos)
    except Exception as e:
        return f"Error al buscar en los documentos: {e}"

In [ ]:
print(buscar_documentos.invoke({"pregunta": "¿Qué microcontrolador usa el Air-Sat?"})[:700])
print("\n--- fuera de alcance ---")
print(buscar_documentos.invoke({"pregunta": "¿Cuál es la capital de Francia?"}))